# Hidden State Initialization 


Main Ideas:
- Spinup: test grid of spinup hours and evaluate effect on forecast accuracy
    - Apply at training step or not?
        - apply spinup at test set step only, keep training unchanged
        - incorporate spinup into training, mask out spinup period
    - Question: can I just test on train 2023 -> test 2024? Or any issues with overfitting to test set
- Direct Initialization
    - input constant data until equilibrium, extract hidden states from equilibrium conditions, store those in table for initialization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import sys
from dateutil.relativedelta import relativedelta
from sklearn.metrics import mean_squared_error
sys.path.append("../src")
from models.moisture_rnn import RNNData, scale_3d
import pandas as pd
from utils import str2time
from data_funcs import cv_data_wrap
from viz import plot_one

In [ ]:
# Read in trained model
rnn = tf.keras.models.load_model("../models/train_rocky/rnn.keras")

In [ ]:
# Read in ml data
dat = pd.read_pickle("../outputs/forecast_outputs/ml_data.pkl")

In [ ]:
features_list = ['Ed', 'Ew', 'solar', 'wind', 'elev', 'lon', 'lat', 'rain', 'hod', 'doy'] 
tstart = pd.to_datetime('2023-01-01 00:00:00+00:00')
tend = pd.to_datetime('2023-12-31 23:00:00+00:00')
train, val, test = cv_data_wrap(dat, fstart=None, fend=None, tstart=tstart, tend=tend, val_hours=48, test_frac = 0.1, random_state=42)
rnndat = RNNData(train, val, test=None, method="random", timesteps=48, random_state=None, features_list = features_list)

In [ ]:
rnndat.scale_data()

## Case Study

In [ ]:
st = 'RLAS2'
fstart = pd.to_datetime('2024-02-08 00:00:00+00:00')
fend = fstart + relativedelta(hours=48-1)

In [ ]:
cond = (dat[st]["data"].date_time >= fstart) & (dat[st]["data"].date_time <= fend)
fm = dat[st]["data"][cond].fm.to_numpy()
X = dat[st]["data"][cond]
X = X[features_list].to_numpy()
X = rnndat.scaler.transform(X)
X = X.reshape(1, -1, 10)

In [ ]:
preds = rnn.predict(X)

In [ ]:
mse = np.round(mean_squared_error(preds.flatten(), fm), 3)
mse

In [ ]:
plot_one(dat, st=st, m=preds.flatten(), start_time = fstart, end_time=fend, title2=f"MSE: {mse}")

## Try Spinup

In [ ]:
spinup = [20, 10, 5, 0]

In [ ]:
st = 'RLAS2'
fstart = pd.to_datetime('2024-02-08 00:00:00+00:00')
fend = fstart + relativedelta(hours=48-1)

In [ ]:
spinups = {}

for sp in spinup:
    print("~"*50)
    print(f"Spinup hours: {sp}")
    fstart_sp = fstart - relativedelta(hours = sp)
    print(f"Spinup start time: {fstart_sp}")
    cond = (dat[st]["data"].date_time >= fstart_sp) & (dat[st]["data"].date_time <= fend)
    fm = dat[st]["data"][cond].fm.to_numpy()
    print(f"{fm.shape=}")
    X = dat[st]["data"][cond]
    X = X[features_list].to_numpy()
    X = rnndat.scaler.transform(X)
    X = X.reshape(1, -1, 10)
    print(f"{X.shape=}")
    preds = rnn.predict(X)
    print(f"{preds.shape=}")
    mse = np.round(mean_squared_error(preds.flatten(), fm), 3)
    spinups[f"{sp}"] = {
        "X": X,
        "fm": fm,
        "preds": preds,
        "mse": mse
    }

In [ ]:
plot_one(dat, st=st, m = spinups["20"]["preds"].flatten(), start_time = fstart - relativedelta(hours = 20), end_time=fend)

In [ ]:
plot_one(dat, st=st, m = spinups["5"]["preds"].flatten(), start_time = fstart - relativedelta(hours = 5), end_time=fend)

### Try constant spinup

use previous hour

In [ ]:
st = 'RLAS2'
fstart = pd.to_datetime('2024-02-08 00:00:00+00:00')
fend = fstart + relativedelta(hours=48-1)

In [ ]:
cond = (dat[st]["data"].date_time >= fstart) & (dat[st]["data"].date_time <= fend)
fm = dat[st]["data"][cond].fm.to_numpy()
X = dat[st]["data"][cond]
X = X[features_list].to_numpy()
X = rnndat.scaler.transform(X)
X = X.reshape(1, -1, 10)

In [ ]:
cond = (dat[st]["data"].date_time == fstart-relativedelta(hours=1))
Xs = dat[st]["data"][cond]
Xs = Xs[features_list].to_numpy()
Xs = rnndat.scaler.transform(Xs)

nt = 24*7
Xs = np.repeat(Xs, nt, axis=0)
Xs = Xs.reshape(1, -1, 10)

In [ ]:
ts = np.arange(0, nt)
pred = rnn.predict(Xs)
plt.plot(ts, pred.flatten())

In [ ]:
XX = np.concatenate((Xs, X), axis=1)
pred = rnn.predict(XX)

In [ ]:
plt.plot(pred.flatten())

In [ ]:
pf = pred.flatten()
pf = pf[-48:]

In [ ]:
plot_one(dat, st=st, m = pf, start_time = fstart, end_time=fend)